In [1]:
# ================================================================ 
import os 
import joblib 
import pandas as pd 
from sklearn.datasets import load_iris 
from sklearn.preprocessing import StandardScaler 
 
from sklearn.model_selection import train_test_split 
from sklearn.ensemble import RandomForestClassifier 
from sklearn.metrics import accuracy_score, classification_report 
  
  
class DataPipeline: 
    """A minimal but complete automated ML data pipeline.""" 
  
    def __init__(self, artifact_dir="artifacts"): 
        self.artifact_dir = artifact_dir 
        self.scaler = StandardScaler() 
        self.model = RandomForestClassifier(n_estimators=100, random_state=42) 
        os.makedirs(artifact_dir, exist_ok=True) 
  
    # ---- Stage 1: Extract ---- 
    def extract(self): 
        data = load_iris(as_frame=True) 
        df = data.frame 
        print(f"[EXTRACT ] loaded {df.shape[0]} rows x {df.shape[1]} cols") 
        return df 
  
    # ---- Stage 2: Validate ---- 
    def validate(self, df): 
        missing = int(df.isnull().sum().sum()) 
        assert missing == 0, "Data contains missing values!" 
        assert "target" in df.columns, "Target column missing!" 
        print(f"[VALIDATE] no missing values, target present -> OK") 
        return df 
  
    # ---- Stage 3: Transform ---- 
    def transform(self, df): 
        X = df.drop(columns=["target"]) 
        y = df["target"] 
        X_scaled = self.scaler.fit_transform(X) 
        print(f"[TRANSFORM] standard-scaled {X.shape[1]} features") 
        return X_scaled, y 
  
    # ---- Stage 4 + 5: Train and Evaluate ---- 
    def train(self, X, y): 
        X_tr, X_te, y_tr, y_te = train_test_split( 
            X, y, test_size=0.2, random_state=42, stratify=y) 
        self.model.fit(X_tr, y_tr) 
        y_pred = self.model.predict(X_te) 
        acc = accuracy_score(y_te, y_pred) 
        print(f"[TRAIN   ] model fitted, test accuracy = {acc:.3f}") 
        print("[REPORT  ] per-class metrics:") 
        print(classification_report(y_te, y_pred, digits=2)) 
        return acc 
  
    # ---- Stage 6: Persist ---- 
    def persist(self): 
        model_path = os.path.join(self.artifact_dir, "model.pkl") 
        scaler_path = os.path.join(self.artifact_dir, "scaler.pkl") 
        joblib.dump(self.model, model_path) 
        joblib.dump(self.scaler, scaler_path) 
        print(f"[PERSIST ] saved model -> {model_path}") 
        print(f"[PERSIST ] saved scaler -> {scaler_path}") 
  
    # ---- Orchestration ---- 
    def run(self): 
        print("=== Automated Data Pipeline Started ===") 
        df = self.extract() 
        df = self.validate(df) 
        X, y = self.transform(df) 
        self.train(X, y) 
 
        self.persist() 
        print("=== Pipeline Completed Successfully ===") 
  
  
if __name__ == "__main__": 
    DataPipeline().run()

=== Automated Data Pipeline Started ===
[EXTRACT ] loaded 150 rows x 5 cols
[VALIDATE] no missing values, target present -> OK
[TRANSFORM] standard-scaled 4 features
[TRAIN   ] model fitted, test accuracy = 0.933
[REPORT  ] per-class metrics:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       0.90      0.90      0.90        10
           2       0.90      0.90      0.90        10

    accuracy                           0.93        30
   macro avg       0.93      0.93      0.93        30
weighted avg       0.93      0.93      0.93        30

[PERSIST ] saved model -> artifacts\model.pkl
[PERSIST ] saved scaler -> artifacts\scaler.pkl
=== Pipeline Completed Successfully ===


In [6]:
# Experiment 7: Secure ML Prediction API using Flask 
# ================================================================ 
# Adds:  /health (open), /predict (secured), input validation, 
# API-key authentication, and structured JSON error handling. 
# ================================================================ 
import os 
import joblib 
from flask import Flask, request, jsonify 
from sklearn.datasets import load_iris 
 
from sklearn.ensemble import RandomForestClassifier 
  
MODEL_PATH = "iris_model.pkl" 
API_KEY = os.environ.get("API_KEY", "secret123")   # token-based auth 
  
# ---- Train and persist the model once ---- 
if not os.path.exists(MODEL_PATH): 
    X, y = load_iris(return_X_y=True) 
    joblib.dump(RandomForestClassifier(random_state=42).fit(X, y), MODEL_PATH) 
  
model = joblib.load(MODEL_PATH) 
TARGETS = load_iris().target_names 
  
app = Flask(__name__) 
  
  
def authenticate(req): 
    """Return None if authorised, else an (error, status) tuple.""" 
    if req.headers.get("x-api-key") != API_KEY: 
        return {"error": "Unauthorized - invalid API key"}, 401 
    return None 
  
  
def validate(payload): 
    """Validate the incoming JSON body.""" 
    if not payload or "features" not in payload: 
        return {"error": "Missing 'features' in request body"}, 400 
    feats = payload["features"] 
    if not isinstance(feats, list) or len(feats) != 4: 
        return {"error": "'features' must be a list of 4 numbers"}, 400 
    if not all(isinstance(v, (int, float)) for v in feats): 
        return {"error": "All features must be numeric"}, 400 
    return None 
  
  
@app.route("/health", methods=["GET"]) 
def health(): 
    return jsonify({"status": "healthy", "model": "iris-rf"}) 
  
  
@app.route("/predict", methods=["POST"]) 
def predict(): 
    auth_error = authenticate(request) 
    if auth_error: 
        return jsonify(auth_error[0]), auth_error[1] 
  
    payload = request.get_json(silent=True) 
    val_error = validate(payload) 
    if val_error: 
        return jsonify(val_error[0]), val_error[1] 
  
    pred = int(model.predict([payload["features"]])[0]) 
    proba = model.predict_proba([payload["features"]])[0] 
    return jsonify({ 
        "prediction": pred, 
        "class": str(TARGETS[pred]), 
        "confidence": round(float(max(proba)), 3), 
    }) 
  
  
if __name__ == "__main__": 
    client = app.test_client() 
 
    print("=== Secure Flask Prediction API ===") 
  
    r = client.get("/health") 
    print(f"GET  /health              -> {r.status_code} {r.get_json()}") 
  
    r = client.post("/predict", json={"features": [5.1, 3.5, 1.4, 0.2]}) 
    print(f"POST /predict (no key)    -> {r.status_code} {r.get_json()}") 
  
    r = client.post("/predict", json={"features": [5.1, 3.5, 1.4, 0.2]}, 
                    headers={"x-api-key": "wrong"}) 
    print(f"POST /predict (bad key)   -> {r.status_code} {r.get_json()}") 
  
    r = client.post("/predict", json={"features": [1.4]}, 
                    headers={"x-api-key": API_KEY}) 
    print(f"POST /predict (bad input) -> {r.status_code} {r.get_json()}") 
  
    r = client.post("/predict", json={"features": [6.7, 3.0, 5.2, 2.3]}, 
                    headers={"x-api-key": API_KEY}) 
    print(f"POST /predict (valid)     -> {r.status_code} {r.get_json()}") 

=== Secure Flask Prediction API ===
GET  /health              -> 200 {'model': 'iris-rf', 'status': 'healthy'}
POST /predict (no key)    -> 401 {'error': 'Unauthorized - invalid API key'}
POST /predict (bad key)   -> 401 {'error': 'Unauthorized - invalid API key'}
POST /predict (bad input) -> 400 {'error': "'features' must be a list of 4 numbers"}
POST /predict (valid)     -> 200 {'class': 'virginica', 'confidence': 1.0, 'prediction': 2}
